# 02. 여러 번 학습 + 지름길 고치기 실험

01번 노트북의 결과(검은 띠 지름길)를 **더 단단하게** 확인하고, **고치는 방법**까지 비교합니다.

| 실험 | 학습 사진 | 묻는 질문 |
|---|---|---|
| **clean** | 늘림(띠 없음) | 기준 |
| **padded** | 검은 띠 (letterbox) | 01번과 같은 지름길 조건 |
| **randband** | 띠 두께를 **무작위**로 (정상·폐렴 모두) | 띠 두께와 정답의 연결을 끊으면 지름길이 사라지나? |
| **grayfill** | 띠를 검은색 대신 **사진 평균 밝기(회색)**로 | 띠가 눈에 덜 띄면 덜 속나? |

각 실험을 **시드 3개(42, 7, 2024)**로 학습해 숫자의 흔들림(평균 ± 표준편차)도 봅니다. 총 12번 학습, T4 기준 약 40~60분.

**시작 전:** `런타임 → 런타임 유형 변경 → T4 GPU`, 그다음 `런타임 → 모두 실행`.

> 결과는 드라이브 `xray-gradcam-story/outputs/robust/`에 저장됩니다. 중간에 끊겨도 끝난 실험은 다시 학습하지 않고 건너뜁니다.

In [ ]:
# ===== 1. Mount Google Drive =====


# google.colab.drive: 구글 드라이브를 Colab 폴더처럼 연결하는 도구
from google.colab import drive
drive.mount('/content/drive')


# → 드라이브를 연결해 데이터 zip을 읽고 결과를 저장하겠다

In [ ]:
# ===== 2. Paths & unzip =====


import os, glob
DRIVE_DIR = '/content/drive/MyDrive/xray-gradcam-story'
ZIP_PATH = f'{DRIVE_DIR}/data/ChestXRay2017.zip'
WORK_DIR = '/content/data'
# OUT_DIR: 이번 노트북 결과 폴더 (01번 outputs와 섞이지 않게 하위 폴더 사용)
OUT_DIR = f'{DRIVE_DIR}/outputs/robust'
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(WORK_DIR):
    !unzip -q "{ZIP_PATH}" -d "{WORK_DIR}"
TRAIN_DIR = [d for d in glob.glob(f'{WORK_DIR}/**/train', recursive=True) if '__MACOSX' not in d][0]
TEST_DIR = os.path.join(os.path.dirname(TRAIN_DIR), 'test')
print(TRAIN_DIR, TEST_DIR)


# → 01번과 같은 방식으로 데이터를 Colab 디스크에 풀겠다

In [ ]:
# ===== 3. Imports =====


import re, json, random, time
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
print('device:', DEVICE)

# CFG: 01번 노트북과 같은 학습 설정 (비교가 공정하도록)
CFG = dict(img_size=224, batch_size=32, epochs=5, lr=1e-4, val_folds=5, split_seed=42,
           seeds=[42, 7, 2024], variants=['clean', 'padded', 'randband', 'grayfill'],
           # randband: 학습 때 사진 내용 높이를 원래 높이의 55%~100% 사이에서 무작위로 줄이고 위아래를 검게 채움
           randband_min=0.55)
S = CFG['img_size']
print(CFG)


# → 필요한 도구와 실험 설정을 준비하겠다 (seed는 학습마다 따로 고정)

In [ ]:
# ===== 4. File list & patient-level split (same split for every run) =====


def list_images(folder):
    return sorted(p for p in glob.glob(f'{folder}/*') if p.lower().endswith(('.jpeg', '.jpg', '.png')))

def patient_id(path):
    name = os.path.basename(path)
    m = re.match(r'(person\d+)_', name)
    if m: return m.group(1)
    m = re.match(r'((?:NORMAL2-)?IM-\d+)-', name)
    if m: return m.group(1)
    return name

def build_table(root):
    rows = []
    for label, cls in enumerate(['NORMAL', 'PNEUMONIA']):
        for p in list_images(f'{root}/{cls}'):
            rows.append(dict(path=p, label=label, patient=patient_id(p)))
    return pd.DataFrame(rows)

df_all, df_test = build_table(TRAIN_DIR), build_table(TEST_DIR)
# 분할은 시드와 상관없이 split_seed 하나로 고정 → 실험끼리 같은 학습/검증 사진을 씀
sgkf = StratifiedGroupKFold(n_splits=CFG['val_folds'], shuffle=True, random_state=CFG['split_seed'])
tr_idx, va_idx = next(sgkf.split(df_all, df_all['label'], groups=df_all['patient']))
df_tr, df_va = df_all.iloc[tr_idx].reset_index(drop=True), df_all.iloc[va_idx].reset_index(drop=True)
print('train', len(df_tr), 'val', len(df_va), 'test', len(df_test))


# → 01번과 똑같이 환자 단위로 나누겠다 (모든 실험이 같은 분할을 쓰도록)

In [ ]:
# ===== 5. Preprocess: stretch / letterbox(black) / letterbox(gray) =====


def load_gray(p):
    im = Image.open(p); im.draft('L', (S * 2, S * 2)); return im.convert('L')

def stretch(im):
    return np.asarray(im.resize((S, S), Image.BILINEAR))

def letterbox(im, fill=0):
    # fill: 남는 칸을 채울 밝기. 0이면 검은색, 'mean'이면 그 사진의 평균 밝기(회색)
    w, h = im.size; r = S / max(w, h)
    small = im.resize((max(1, round(w * r)), max(1, round(h * r))), Image.BILINEAR)
    v = int(np.asarray(small).mean()) if fill == 'mean' else fill
    canvas = Image.new('L', (S, S), v)
    canvas.paste(small, ((S - small.width) // 2, (S - small.height) // 2))
    return np.asarray(canvas)

def build_arrays(paths):
    X = {k: np.zeros((len(paths), S, S), np.uint8) for k in ['st', 'lb', 'gr']}
    for i, p in enumerate(tqdm(list(paths), desc='preprocess')):
        im = load_gray(p)
        X['st'][i] = stretch(im); X['lb'][i] = letterbox(im); X['gr'][i] = letterbox(im, 'mean')
    return X

Xtr, Xva, Xte = build_arrays(df_tr.path), build_arrays(df_va.path), build_arrays(df_test.path)
ytr, yva, yte = df_tr.label.values, df_va.label.values, df_test.label.values


# → 사진마다 '늘림 / 검은 띠 / 회색 띠' 세 가지 224×224 버전을 미리 만들어 두겠다

In [ ]:
# ===== 6. Dataset (randband is applied on the fly) =====


MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def random_band(x):
    # x: [1,S,S] 늘린 사진. 내용 높이를 무작위로 줄이고 위아래를 0(검정)으로 채운다
    # → 띠 두께가 정상/폐렴과 상관없이 매번 달라져서, 띠로 정답을 맞힐 수 없게 된다
    r = random.uniform(CFG['randband_min'], 1.0)
    h = max(8, int(round(S * r)))
    small = F.interpolate(x[None], size=(h, S), mode='bilinear', align_corners=False)[0]
    out = torch.zeros_like(x); top = (S - h) // 2
    out[:, top:top + h] = small
    return out

class XrayDS(Dataset):
    def __init__(self, X, y, train=False, randband=False):
        self.X, self.y, self.train, self.randband = X, y, train, randband
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]).float().div(255).unsqueeze(0)
        if self.train:
            x = (x * random.uniform(0.9, 1.1) + random.uniform(-0.05, 0.05)).clamp(0, 1)
            if self.randband: x = random_band(x)
        x = (x.repeat(3, 1, 1) - MEAN) / STD
        return x, int(self.y[i])

def make_loader(X, y, train, randband=False):
    return DataLoader(XrayDS(X, y, train, randband), batch_size=CFG['batch_size'], shuffle=train,
                      num_workers=2, pin_memory=(DEVICE == 'cuda'))


# → randband 실험은 학습할 때마다 띠 두께를 새로 뽑아 붙이는 데이터 공급기를 만들겠다

In [ ]:
# ===== 7. Train / predict =====


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, X):
    model.eval(); out = []
    for xb, _ in make_loader(X, np.zeros(len(X)), train=False):
        with torch.autocast(device_type='cuda', enabled=USE_AMP):
            o = model(xb.to(DEVICE))
        out.append(o.float().softmax(1)[:, 1].cpu().numpy())
    return np.concatenate(out)

# 실험별로 학습·검증에 쓸 사진 버전
TRAIN_VIEW = {'clean': 'st', 'padded': 'lb', 'randband': 'st', 'grayfill': 'gr'}

def train_run(variant, seed):
    set_seed(seed)
    view, rb = TRAIN_VIEW[variant], variant == 'randband'
    model = build_model()
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best, best_state = -1, None
    # randband 검증은 검은 띠 버전으로 (띠가 있어도 흔들리지 않는지)
    Xv = Xva['lb'] if rb else Xva[view]
    for ep in range(CFG['epochs']):
        model.train()
        for xb, yb in make_loader(Xtr[view], ytr, train=True, randband=rb):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=USE_AMP):
                loss = F.cross_entropy(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        auc = roc_auc_score(yva, predict(model, Xv))
        if auc > best: best, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best

def score(p, y, th=0.5):
    pred = (p >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(acc=float((tp + tn) / len(y)), auc=float(roc_auc_score(y, p)), FN=int(fn), FP=int(fp))


# → 시드를 고정해 한 번 학습하고, 검증 AUC가 가장 좋았던 가중치를 돌려주는 함수를 만들겠다

In [ ]:
# ===== 8. Run all experiments (resumable) =====


RUNS_PATH = f'{OUT_DIR}/runs.json'
runs = json.load(open(RUNS_PATH)) if os.path.exists(RUNS_PATH) else []
done = {(r['variant'], r['seed']) for r in runs}

for variant in CFG['variants']:
    for seed in CFG['seeds']:
        if (variant, seed) in done:
            print('skip', variant, seed); continue
        t0 = time.time()
        model, val_auc = train_run(variant, seed)
        p_plain, p_pad = predict(model, Xte['st']), predict(model, Xte['lb'])
        r = dict(variant=variant, seed=seed, val_auc=float(val_auc),
                 plain=score(p_plain, yte), padded=score(p_pad, yte),
                 probs_plain=np.round(p_plain, 4).tolist(), probs_padded=np.round(p_pad, 4).tolist())
        runs.append(r)
        json.dump(runs, open(RUNS_PATH, 'w'))
        if seed == 42 and variant in ('randband', 'grayfill'):
            torch.save(model.state_dict(), f'{OUT_DIR}/{variant}_s42.pth')
        print(f"{variant:9s} s{seed:<5d} plain FN={r['plain']['FN']:3d} acc={r['plain']['acc']:.4f} | padded FN={r['padded']['FN']:3d} acc={r['padded']['acc']:.4f}  ({time.time() - t0:.0f}s)")
        del model; torch.cuda.empty_cache()


# → 4가지 실험 × 시드 3개를 차례로 학습하고, 끝날 때마다 runs.json에 저장하겠다 (끊겨도 이어서 가능)

In [ ]:
# ===== 9. Summary table (mean ± std over seeds) =====


rows = []
for r in runs:
    for test in ['plain', 'padded']:
        rows.append(dict(variant=r['variant'], seed=r['seed'], test=test, **r[test]))
df = pd.DataFrame(rows)
summary = df.groupby(['variant', 'test'])[['acc', 'auc', 'FN', 'FP']].agg(['mean', 'std']).round(3)
summary.to_csv(f'{OUT_DIR}/summary.csv')
summary


# → 실험·시험 조건별로 정확도, AUC, FN, FP의 평균과 표준편차를 정리하겠다

### 결과 읽는 법

- **padded 모델**에서 `plain`(띠 없는 시험)의 FN이 `padded`(띠 있는 시험)보다 크게 많으면 → 01번과 같은 지름길이 시드를 바꿔도 재현된 것
- **randband**에서 두 시험의 FN 차이가 작아지면 → 띠 두께와 정답의 연결을 끊어 지름길이 약해진 것
- **grayfill**은 띠를 눈에 덜 띄게 했을 때의 효과
- 표준편차(std)가 크면 그 숫자는 학습마다 많이 흔들린다는 뜻

## 다음 단계

드라이브 `xray-gradcam-story/outputs/robust/`에서 아래 파일을 받아 로컬 `xray-gradcam-story/data/`에 넣어 주세요.

- `runs.json` (모든 실험 결과와 시험 사진 확률)
- `randband_s42.pth`, `grayfill_s42.pth` (근거 지도 비교용 가중치)